# Aprire la scatola nera

Il codice del capitolo [«Aprire la scatola nera»](https://book.paithon.it/main/Interpretabilita/overview.html), *Paithon Book*.

Le celle sono quelle del libro, nell'ordine in cui compaiono: il testo che le spiega sta nelle pagine, qui c'è solo la parte da eseguire e da rompere.

Generato da `scripts/genera-notebook.py`: le correzioni vanno fatte nelle pagine del libro, non qui.


> **Verificato il 2026-07-25** con torch 2.13.0, numpy 2.4.6, pandas 3.0.5, scikit-learn 1.9.0, transformers 5.14.1, diffusers 0.39.0, librosa 0.11.0, torch-geometric 2.8.0.post1. Tutte le celle di questo notebook sono state eseguite senza errori con quelle versioni; le librerie si muovono, e se qualcosa qui non gira piu' e' un errore del libro: [segnalalo](https://github.com/paithon-it/paithonbook/issues).


In [ ]:
# Su Colab quasi tutto c'è già; questa riga serve altrove.
%pip install -q numpy scikit-learn torch torchvision

In [ ]:
# Mostra il valore di ogni riga, come i commenti «# ->» del libro.
try:
    from IPython.core.interactiveshell import InteractiveShell
    InteractiveShell.ast_node_interactivity = 'all'
except ImportError:      # fuori da IPython non serve e non c'è
    pass

## Aprire la scatola nera

[Leggi la pagina](https://book.paithon.it/main/Interpretabilita/overview.html)


### Un modello che si spiega da sé


In [ ]:
from sklearn.datasets import load_iris
from sklearn.tree import DecisionTreeClassifier, export_text

# Un modello intrinsecamente interpretabile: un albero volutamente basso
iris = load_iris()
X, y = iris.data, iris.target
albero = DecisionTreeClassifier(max_depth=2, random_state=0)
albero.fit(X, y)

# Tutta la "logica" del modello è leggibile come una ricetta di if-then
print(export_text(albero, feature_names=list(iris.feature_names)))

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# dieci gruppi: nove per imparare, uno per l'esame, e si gira dieci volte
foresta = RandomForestClassifier(n_estimators=300, random_state=0)
for nome, m in [("alberello", albero), ("foresta casuale", foresta)]:
    print(f"{nome:16} {cross_val_score(m, X, y, cv=10).mean():.1%}")

## Modelli trasparenti e importanza delle feature

[Leggi la pagina](https://book.paithon.it/main/Interpretabilita/modelli-trasparenti-e-importanza.html)


### In pratica: rimescolamento contro impurità


In [ ]:
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance

dati = load_diabetes()
X, y, nomi = dati.data, dati.target, list(dati.feature_names)

# Due colonne di puro rumore, scorrelate dal target: una continua e una binaria.
# Non valgono niente ne l'una ne l'altra: servono da metro per le due misure.
rng = np.random.default_rng(0)
X = np.column_stack([X, rng.normal(size=len(y)), rng.integers(0, 2, size=len(y))])
nomi += ["rumore_cont", "rumore_bin"]

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0)

rf = RandomForestRegressor(n_estimators=300, random_state=0)
rf.fit(X_tr, y_tr)
print("R^2 sul test:", round(rf.score(X_te, y_te), 3))  # -> 0.315

# Importanza da permutazione, misurata sul TEST (10 mescolamenti per feature)
pi = permutation_importance(rf, X_te, y_te, n_repeats=10, random_state=0)

print("feature      (impurita)   perm-import   valori distinti")
for i in np.argsort(rf.feature_importances_)[::-1]:   # dalla piu alta per la MDI
    print(f"{nomi[i]:>11}   {rf.feature_importances_[i]:.4f}      "
          f"{pi.importances_mean[i]:+.3f} +/- {pi.importances_std[i]:.3f}"
          f"   {len(np.unique(X_tr[:, i])):5d}")

## Spiegazioni locali: LIME, SHAP e controfattuali

[Leggi la pagina](https://book.paithon.it/main/Interpretabilita/spiegazioni-locali.html)


### In pratica: i valori di Shapley calcolati da zero


In [ ]:
import itertools
from math import factorial
import numpy as np

# Un modello giocattolo con un'interazione: la feature 2 "conta" solo con la 0
def f(x):
    return x[0] + 2.0 * x[1] + x[0] * x[2]

# istanza da spiegare e riferimento (baseline) su cui "spegnere" le feature assenti
x = np.array([1.0, 1.0, 1.0])
r = np.array([0.0, 0.0, 0.0])
n = len(x)

# valore della coalizione S: le feature in S prendono il valore di x, le altre di r
def v(S):
    z = r.copy()
    for i in S:
        z[i] = x[i]
    return f(z)

# valori di Shapley per forza bruta: media dei contributi marginali su TUTTI gli ordini
phi = np.zeros(n)
for perm in itertools.permutations(range(n)):
    S = []
    for i in perm:
        prima = v(S)            # coalizione prima di aggiungere i
        S = S + [i]
        dopo = v(S)             # coalizione dopo aver aggiunto i
        phi[i] += dopo - prima  # contributo marginale di i in questo ordine
phi /= factorial(n)             # media sugli n! ordini

print("valori di Shapley:", np.round(phi, 3))
print("somma dei phi:     ", round(float(phi.sum()), 3))
print("f(x) - f(base):    ", round(float(f(x) - f(r)), 3))  # assioma di efficienza

## Dentro le reti profonde: attribuzione e interpretabilità meccanicistica

[Leggi la pagina](https://book.paithon.it/main/Interpretabilita/attribuzione-e-meccanicistica.html)


### Integrated Gradients coi numeri: un esempio eseguibile


In [ ]:
import numpy as np

# funzione giocattolo che satura: f(x) = tanh(w . x)
w = np.array([2.0, -1.0])

def f(x):
    return np.tanh(w @ x)

def grad_f(x):
    z = w @ x
    return (1.0 - np.tanh(z) ** 2) * w   # regola della catena

x = np.array([2.0, 1.0])     # input da spiegare
baseline = np.zeros(2)        # baseline neutra (lo "zero")

# gradiente grezzo nel solo punto x: saturo, quasi nullo -> saliency cieca
print("gradiente in x :", np.round(grad_f(x), 4))       # [ 0.0197 -0.0099]

# Integrated Gradients: media dei gradienti lungo il cammino baseline -> x
m = 200
alphas = (np.arange(1, m + 1) - 0.5) / m   # punti medi delle m tappe
grad_medio = np.zeros(2)
for a in alphas:
    grad_medio += grad_f(baseline + a * (x - baseline))
grad_medio /= m
ig = (x - baseline) * grad_medio
print("attribuzioni IG:", np.round(ig, 4))              # [ 1.3267 -0.3317]

# assioma di completezza: la somma delle attribuzioni = f(x) - f(baseline)
print("somma IG       :", round(ig.sum(), 4))           # 0.9951
print("f(x) - f(base) :", round(f(x) - f(baseline), 4)) # 0.9951

### Uno sketch di Grad-CAM in PyTorch


In [ ]:
import torch
import torch.nn.functional as F
from torchvision import models

model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1).eval()
target = model.layer4[-1]                # ultimo blocco: uscita post-residuo

att, grad = {}, {}
target.register_forward_hook(lambda m, i, o: att.__setitem__("v", o.detach()))
target.register_full_backward_hook(
    lambda m, gi, go: grad.__setitem__("v", go[0].detach())
)

x = torch.randn(1, 3, 224, 224)          # immagine gia pre-processata
logit = model(x)                          # (1, 1000)
classe = logit.argmax(dim=1)              # classe predetta
model.zero_grad()
logit[0, classe].backward()               # gradiente della sola classe scelta

A = att["v"]                              # attivazioni  (1, C, h, w)
dY = grad["v"]                            # gradienti    (1, C, h, w)
alpha = dY.mean(dim=(2, 3), keepdim=True)  # peso per canale (global avg pool)
heatmap = F.relu((alpha * A).sum(dim=1))   # (1, h, w), solo contributi positivi
heatmap = heatmap / (heatmap.max() + 1e-8) # normalizzata in [0, 1]
# heatmap va poi sovracampionata a 224x224 e sovrapposta all'immagine

print("attivazioni:", tuple(A.shape))
print("gradienti  :", tuple(dY.shape))
print("heatmap    :", tuple(heatmap.shape))